# Milestone 2: Data Cleaning & Exploratory Data Analysis  
## Student Performance Analytics

**Course deliverable:** Data Cleaning & EDA Notebook  
**Dataset used:** `student_data.csv` provided for this project

### Objective
This notebook analyzes student academic performance to identify patterns related to grades, attendance, study habits, family background, and lifestyle factors. The goals are to:

- document a reproducible data loading and cleaning pipeline,
- perform structured exploratory data analysis,
- extract evidence-based insights about student success and risk factors,
- propose hypotheses for deeper analysis in later milestones.

### Research Questions
1. Which student behaviors and background factors are most associated with final grades?
2. How do attendance, prior failures, and study time relate to academic outcomes?
3. Are there meaningful group differences across categories such as school support, internet access, or family education?

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10

sns.set_theme(style="whitegrid")

## 1. Data Loading

The notebook loads the project dataset from a relative path so another student can reproduce the analysis without changing machine-specific file locations.

In [ ]:
possible_paths = [
    Path("student_data.csv"),
    Path("/mnt/data/student_data.csv")
]

data_path = None
for path in possible_paths:
    if path.exists():
        data_path = path
        break

if data_path is None:
    raise FileNotFoundError("Could not find student_data.csv in the current folder or /mnt/data.")

df_raw = pd.read_csv(data_path)
df_raw.head()

**Interpretive note:**  
The dataset contains student demographic, academic, support, and lifestyle variables, along with three grade variables (`G1`, `G2`, `G3`). `G3` is the final grade and serves as the main outcome of interest.

## 2. Initial Inspection
We inspect shape, column types, missingness, and duplicates before applying any transformations.

In [ ]:
print("Shape:", df_raw.shape)
print("\nColumn names:")
print(df_raw.columns.tolist())

print("\nData types:")
display(df_raw.dtypes)

print("\nMissing values per column:")
display(df_raw.isna().sum().sort_values(ascending=False))

print("\nDuplicate rows:", df_raw.duplicated().sum())

**Interpretive note:**  
The dataset has **395 rows and 33 columns**. It contains a mix of categorical and numeric features. No missing values and no duplicate rows were detected, which means the cleaning work here focuses more on **type handling, consistency checks, and feature engineering** than on heavy imputation.

## 3. Data Cleaning and Preprocessing Pipeline

A strong cleaning workflow should still be documented even when the raw file is relatively clean. The steps below are intentionally explicit for reproducibility and grading transparency:

1. create a working copy,
2. verify ranges and data integrity,
3. map binary yes/no fields to cleaner labels where useful,
4. derive analysis-friendly features,
5. create encoded data only when needed for correlation analysis.

In [ ]:
df = df_raw.copy()

# Sanity checks for grade ranges based on the dataset's grading scheme (0-20)
grade_cols = ["G1", "G2", "G3"]
for col in grade_cols:
    assert df[col].between(0, 20).all(), f"{col} contains values outside expected range 0-20."

# Sanity checks for selected ordinal/count variables
assert (df["absences"] >= 0).all(), "Absences contains negative values."
assert (df["failures"] >= 0).all(), "Failures contains negative values."

# Standardize binary yes/no fields into readable labels for presentation
binary_cols = ["schoolsup", "famsup", "paid", "activities", "nursery", "higher", "internet", "romantic"]
for col in binary_cols:
    df[col] = df[col].map({"yes": "Yes", "no": "No"})

df.head()

**Cleaning justification:**  
Although the dataset did not require missing-value imputation or duplicate removal, the notebook still performs **data validation and standardization**, which is an important part of preprocessing. This prevents silent errors later and makes the dataset easier to interpret visually.

## 4. Feature Engineering

To move beyond basic EDA, we create derived variables that better reflect academic risk and student context.

### Engineered features
- **avg_grade**: average of G1, G2, and G3
- **grade_change**: final grade minus first-period grade
- **parent_edu_avg**: average parental education level
- **total_alcohol**: weekday + weekend alcohol consumption
- **is_high_absence**: flag for students above the 75th percentile in absences
- **performance_band**: categorical grouping of final grade

In [ ]:
df["avg_grade"] = df[["G1", "G2", "G3"]].mean(axis=1)
df["grade_change"] = df["G3"] - df["G1"]
df["parent_edu_avg"] = df[["Medu", "Fedu"]].mean(axis=1)
df["total_alcohol"] = df["Dalc"] + df["Walc"]

absence_threshold = df["absences"].quantile(0.75)
df["is_high_absence"] = np.where(df["absences"] > absence_threshold, "High absence", "Lower absence")

df["performance_band"] = pd.cut(
    df["G3"],
    bins=[-0.1, 9.99, 13.99, 16.99, 20],
    labels=["Low (0-9)", "Moderate (10-13)", "Good (14-16)", "High (17-20)"]
)

df[["G1", "G2", "G3", "avg_grade", "grade_change", "parent_edu_avg", "total_alcohol", "is_high_absence", "performance_band"]].head()

**Interpretive note:**  
These derived features make the analysis more meaningful. For example, `grade_change` captures academic trajectory rather than just a single score, while `performance_band` supports clearer group comparisons than raw grades alone.

## 5. Summary Statistics and Distributional Reasoning

Before plotting, we compute descriptive statistics for key outcome and behavioral variables.

In [ ]:
summary_stats = df[["G1", "G2", "G3", "studytime", "absences", "failures", "avg_grade", "grade_change"]].describe().T
display(summary_stats)

In [ ]:
print("Skewness of selected numeric features:")
display(df[["G3", "studytime", "absences", "failures", "grade_change"]].skew().to_frame("skewness"))

**Statistical note:**  
`absences` and `failures` are right-skewed, meaning most students have low values but a smaller group has much higher values. This is important because averages alone can hide concentration of academic risk in a minority of students. `G3` is less skewed and more centrally distributed, making it suitable for both distribution and group-comparison analysis.

## 6. Exploratory Data Analysis

The following visualizations are designed to cover:
- distributions,
- relationships,
- group differences,
- correlations,
- and anomaly/risk patterns.

Each chart includes a short interpretive caption.

In [ ]:
plt.figure()
sns.histplot(df["G3"], bins=12, kde=True)
plt.title("Distribution of Final Grades (G3)")
plt.xlabel("Final Grade")
plt.ylabel("Number of Students")
plt.show()

**Figure 1 caption:**  
Final grades are concentrated in the middle range, with fewer students at the very top and bottom. This suggests a mostly moderate-performing population rather than a heavily polarized one.

In [ ]:
plt.figure()
sns.boxplot(x=df["G3"])
plt.title("Boxplot of Final Grades (G3)")
plt.xlabel("Final Grade")
plt.show()

**Figure 2 caption:**  
The boxplot shows moderate spread and a few low-end outliers. Those lower-score observations may represent academically at-risk students and warrant closer examination in later stages.

In [ ]:
plt.figure()
sns.scatterplot(data=df, x="studytime", y="G3", alpha=0.75)
plt.title("Study Time vs Final Grade")
plt.xlabel("Study Time (ordinal scale)")
plt.ylabel("Final Grade")
plt.show()

**Figure 3 caption:**  
Students with higher study-time categories tend to appear more often in the stronger grade range, though the relationship is not perfectly linear. This suggests study time matters, but other variables also shape outcomes.

In [ ]:
plt.figure()
sns.scatterplot(data=df, x="absences", y="G3", alpha=0.75)
plt.title("Absences vs Final Grade")
plt.xlabel("Absences")
plt.ylabel("Final Grade")
plt.show()

**Figure 4 caption:**  
Higher absence counts are generally associated with lower final grades, with substantial variability. The spread implies absences are an important risk signal, even if they do not fully determine performance on their own.

In [ ]:
plt.figure()
order = ["Low (0-9)", "Moderate (10-13)", "Good (14-16)", "High (17-20)"]
sns.boxplot(data=df, x="performance_band", y="absences", order=order)
plt.title("Absences by Performance Band")
plt.xlabel("Performance Band")
plt.ylabel("Absences")
plt.xticks(rotation=10)
plt.show()

**Figure 5 caption:**  
Lower-performing students tend to have higher and more variable absence counts. This strengthens the hypothesis that attendance is linked to academic success.

In [ ]:
plt.figure()
sns.barplot(data=df, x="failures", y="G3", estimator=np.mean, errorbar=None)
plt.title("Average Final Grade by Number of Past Class Failures")
plt.xlabel("Past Failures")
plt.ylabel("Average Final Grade")
plt.show()

**Figure 6 caption:**  
Average final grade decreases sharply as the number of past failures rises. Prior failure history appears to be one of the clearest warning indicators in the dataset.

In [ ]:
plt.figure()
sns.barplot(data=df, x="schoolsup", y="G3", estimator=np.mean, errorbar=None)
plt.title("Average Final Grade by School Support")
plt.xlabel("School Support")
plt.ylabel("Average Final Grade")
plt.show()

**Figure 7 caption:**  
Students receiving school support do not necessarily have higher average grades. A likely explanation is that support is targeted toward struggling students, so this variable may reflect intervention rather than advantage.

In [ ]:
plt.figure()
sns.barplot(data=df, x="internet", y="G3", estimator=np.mean, errorbar=None)
plt.title("Average Final Grade by Internet Access")
plt.xlabel("Internet Access at Home")
plt.ylabel("Average Final Grade")
plt.show()

**Figure 8 caption:**  
Students with internet access show somewhat stronger average performance. This may reflect access to educational resources, but it could also be related to broader socioeconomic differences.

In [ ]:
corr_cols = ["G1", "G2", "G3", "studytime", "absences", "failures", "parent_edu_avg", "famrel", "goout", "total_alcohol", "health"]
corr_matrix = df[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(10, 7))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", square=True)
plt.title("Correlation Heatmap of Key Numeric Features")
plt.show()

**Figure 9 caption:**  
The strongest positive relationships with final grade are the earlier period grades (`G1`, `G2`), which is expected. Negative associations appear for failures and absences, reinforcing their value as risk indicators.

In [ ]:
plt.figure()
sns.boxplot(data=df, x="sex", y="G3")
plt.title("Final Grade Distribution by Sex")
plt.xlabel("Sex")
plt.ylabel("Final Grade")
plt.show()

**Figure 10 caption:**  
The grade distributions for male and female students overlap substantially. Any difference appears much smaller than the effects observed for variables such as failures, absences, and previous grades.

In [ ]:
plt.figure()
sns.barplot(data=df, x="performance_band", y="studytime", order=order, estimator=np.mean, errorbar=None)
plt.title("Average Study Time by Performance Band")
plt.xlabel("Performance Band")
plt.ylabel("Average Study Time")
plt.xticks(rotation=10)
plt.show()

**Figure 11 caption:**  
Higher performance bands tend to report slightly greater study time on average. This does not prove causation, but it supports the idea that study investment and performance move together.

## 7. Focused Statistical Findings

This section converts the visuals into explicit analytical statements supported by summary statistics and correlations.

In [ ]:
corr_with_g3 = df[corr_cols].corr(numeric_only=True)["G3"].sort_values(ascending=False)
display(corr_with_g3.to_frame("correlation_with_G3"))

performance_counts = df["performance_band"].value_counts().sort_index()
display(performance_counts.to_frame("student_count"))

high_absence_grade = df.groupby("is_high_absence")["G3"].agg(["mean", "median", "count"]).sort_values("mean", ascending=False)
display(high_absence_grade)

failure_grade = df.groupby("failures")["G3"].agg(["mean", "median", "count"])
display(failure_grade)

**Key statistical observations:**
- `G2` and `G1` have the strongest positive correlations with final grade, which indicates grade continuity across periods.
- `failures` has one of the strongest negative associations with `G3`, making it a strong academic risk indicator.
- `absences` is also negatively related to final performance, though less strongly than prior grades and failures.
- Students in the **high-absence** group have weaker average outcomes than those with lower absences.

## 8. Initial Findings and Insight Extraction

### Main findings
1. **Academic trajectory matters most.** Earlier grades (`G1`, `G2`) strongly track with final grade, suggesting student performance is persistent over time.
2. **Past failures are a major warning signal.** Students with more past failures tend to have markedly lower final grades.
3. **Attendance matters.** Students with more absences generally perform worse, and lower-performing groups show wider absence distributions.
4. **Study habits matter, but not alone.** Higher study-time categories are associated with stronger performance, though the effect is weaker than prior grades and failures.
5. **Support variables require careful interpretation.** School support does not automatically correspond to higher grades because support may be reactive rather than preventive.
6. **Demographic differences appear weaker than behavioral/academic ones.** Variables related to attendance, prior performance, and failure history appear more informative than broad demographic splits.

### Non-obvious insight
One especially important pattern is that some support-related variables may look weak or even counterintuitive because they likely identify students who were already struggling. This means interpretation must consider **selection effects**, not just raw group averages.

## 9. Hypotheses for Further Exploration

Based on the EDA, the following hypotheses are worth testing in later milestones:

1. **Students with more past failures have significantly lower final grades than students with no failures.**
2. **Higher absence levels are associated with lower final grades, even after accounting for study time.**
3. **Parental education may have an indirect relationship with performance through access and support variables rather than a direct large effect.**
4. **A predictive model using prior grades, failures, and absences will outperform a model based only on demographic variables.**
5. **Intervention variables such as school support may reflect pre-existing academic risk, so causal interpretation should be avoided without stronger methods.**

## 10. Assumptions and Limitations

### Assumptions
- The provided CSV accurately represents the source dataset.
- The ordinal variables (for example `studytime`, `traveltime`, and alcohol-use scales) can be treated as ordered numeric indicators for basic EDA.
- Final grade `G3` is an appropriate primary measure of student performance.

### Limitations
- This dataset is relatively small, with 395 observations.
- Several variables are categorical or ordinal, so simple correlations may understate nonlinear patterns.
- Some variables may be proxies for more complex social or academic conditions.
- Because this is observational data, the notebook identifies **associations**, not causal effects.
- The dataset appears to capture a specific educational context, so findings may not generalize to all schools or populations.

## 11. Reproducibility and Code Quality Notes

This notebook was structured to support reproducibility:

- uses relative/fallback paths instead of machine-specific hardcoded paths,
- separates raw data from processed data via `df_raw` and `df`,
- documents cleaning and feature-engineering decisions,
- uses readable chart titles and axis labels,
- keeps analysis steps ordered from loading → cleaning → EDA → findings.

### How to run
1. Place `student_data.csv` in the same folder as this notebook, or keep it in `/mnt/data/`.
2. Install dependencies if needed:
   ```bash
   pip install pandas numpy matplotlib seaborn
   ```
3. Run the notebook top to bottom.

### Deliverable summary
This notebook includes:
- a data loading and cleaning pipeline,
- more than 8 static EDA visualizations,
- interpretive captions for each visualization,
- statistical reasoning and summary tables,
- initial findings and hypotheses for further exploration.